In [ ]:
import functools as ft
import pathlib
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)



### Cathether case

In [ ]:
case_name = "catheter_k_002_left"
target_file = pathlib.Path(f"crops/data/targets/{case_name}.csv")
stl_file = pathlib.Path(f"crops/data/obstacles/{case_name}.stl")

# Catheter tube, SI units.
outer_radius = 0.5e-3
inner_radius = 0.35e-3
shear_modulus = 3.03e5
num_elems = 120
num_fibers = 8
sheath_fraction = 0.1  # length still outside at full insertion, fraction of path
steps_per_elem = 3.5

contact_grid_spacing = 0.2e-3
contact_penalty_ratio = 1.0  # contact spring / element bending spring

### Centerline of the lumen

In [ ]:
target = pd.read_csv(target_file)
centerline = target[["x", "y", "z"]].to_numpy()
path_length = float(target["s"].iloc[-1])
mouth = centerline[0]
entry_step = centerline[1] - centerline[0]
entry_tangent = entry_step / np.linalg.norm(entry_step)
rod_length = (1.0 + sheath_fraction) * path_length

print(f"path length {1e3 * path_length:.1f} mm, rod length {1e3 * rod_length:.1f} mm")

### Build the rod 

annular rod 

In [ ]:
mean_radius = 0.5 * (outer_radius + inner_radius)
fiber_area = np.pi * (outer_radius**2 - inner_radius**2) / num_fibers
ring_offsets = _geom.annular_fiber_offsets(
  num_elems=num_elems, num_fibers=num_fibers, semi_axis_d1=mean_radius
)

fiber_offsets = jnp.asarray(ring_offsets)
parametric_coords = jnp.linspace(0.0, rod_length, num_elems + 1)
fiber_areas = jnp.full((num_elems, num_fibers), fiber_area)
fiber_mass_density = jnp.full((num_elems, num_fibers), 1100.0)
fiber_centroid_moi = jnp.zeros((num_elems, num_fibers, 2))
least_aligned_axis = np.argmin(np.abs(entry_tangent))
seed_binormal = jnp.eye(3)[least_aligned_axis]

rod_curve = lambda arc: arc * jnp.asarray(entry_tangent)  # noqa: E731

rod = _mesher.build_rod_geometry(
  analytical_curve=rod_curve,
  parametric_coords=parametric_coords,
  fiber_offsets=fiber_offsets,
  fiber_areas=fiber_areas,
  fiber_mass_density=fiber_mass_density,
  fiber_centroid_moi=fiber_centroid_moi,
  seed_binormal=seed_binormal,
)
contact_radius = float(np.max(rod.contact_radii))
print(f"contact radius {1e3 * contact_radius:.3f} mm")

### Rod insertion (BC) and obstacle movement

In [ ]:
num_load_steps = int(np.ceil(steps_per_elem * num_elems / (1.0 + sheath_fraction)))
max_depth = path_length * 0.9  # rod shortcuts the helix, so its tip exits early
insertion_depths = max_depth * np.arange(1, num_load_steps + 1) / num_load_steps

node_arc = np.asarray(rod.arc_coords)
node_path_coords = node_arc[None, :] - rod_length + insertion_depths[:, None]
outside_nodes = [np.flatnonzero(coords <= 0.0) for coords in node_path_coords]

zero_values = [np.zeros(nodes.size) for nodes in outside_nodes]

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
sheath = [
  _bc.DirichletBC(
    values=(dof, zero_values),
    node_targets=outside_nodes,
    name=f"sheath_{dof}",
  )
  for dof in clamped_dofs
]
bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=sheath)
print(f"{num_load_steps} insertion steps of {1e3 * insertion_depths[0]:.3f} mm")

In [ ]:
vessel = trimesh.load(stl_file, force="mesh")
mouth_positions = np.einsum("k,d->kd", rod_length - insertion_depths, entry_tangent)
vessel_translations = jnp.asarray(mouth_positions - mouth)
movement = _contact.ObstacleMovement.from_translations(vessel_translations)

padding = 3.0 * contact_radius
grid_extent = vessel.extents + 2.0 * padding
grid_counts = np.ceil(grid_extent / contact_grid_spacing).astype(int)
grid_shape = tuple(grid_counts)
wall_penalty_stiffness = _contact.compute_self_contact_stiffness(
  rod=rod,
  youngs_modulus=3.0 * shear_modulus,
  stiffness_ratio=contact_penalty_ratio,
)

obstacle = _contact.Obstacle.from_mesh(
  mesh=vessel,
  grid_shape=grid_shape,
  wall_penalty_stiffness=wall_penalty_stiffness,
  padding=padding,
  extrap_fill_value=10.0,
  movement=movement,
)

### Solver settings

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-11,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-10,
}

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings=nonlinear_settings,
  linear_settings=linear_settings,
  verbose=True,
)

### Problem

In [ ]:
problem = _fe.CosseratProblem(
  rod=rod,
  bc=bc,
  nr_solver=nr_solver,
  obstacle=obstacle,
  proximal_factor=1e-1,
)
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)
u0 = jnp.zeros(rod.num_dofs)

### Design vars and their extents

In [ ]:
max_remnant_flux = 0.143  # [T]
max_applied_flux = 2.0e-2  # [T]

remnant_shape = (num_elems,)  # one vector per element, shared by its fibers
applied_shape = (num_load_steps,)  # one field per load step
polar_extent = _utils.Extent(min=0.0, max=np.pi)
azith_extent = _utils.Extent(min=-np.pi, max=np.pi)
remnant_strength_extent = _utils.Extent(min=0.0, max=max_remnant_flux)
applied_strength_extent = _utils.Extent(min=0.0, max=max_applied_flux)

blocks = [
  ("remnant_polar", remnant_shape, polar_extent),
  ("remnant_azith", remnant_shape, azith_extent),
  ("remnant_strength", remnant_shape, remnant_strength_extent),
  ("applied_polar", applied_shape, polar_extent),
  ("applied_azith", applied_shape, azith_extent),
  ("applied_strength", applied_shape, applied_strength_extent),
]
layout = _utils.DesignLayout(blocks=blocks)

In [ ]:
field_filter_radius = 5.0e-3  # [m] of insertion depth
depth_coords = insertion_depths[:, None]
field_filter = _utils.create_density_filter(
  coords=depth_coords,
  cutoff_distance=field_filter_radius,
)


def design_to_fluxes(design_vars: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Maps the flat design vector to remnant (E, F, 3) and applied (K, 3) flux."""
  physical = layout.to_physical(flat=design_vars)
  remnant_per_elem = _utils.polar_azith_to_vector(
    polar_ang=physical["remnant_polar"],
    azith_ang=physical["remnant_azith"],
    vec_mag=physical["remnant_strength"],
  )
  remnant_mag_flux = jnp.broadcast_to(
    remnant_per_elem[:, None, :],
    (num_elems, num_fibers, 3),
  )
  raw_applied_mag_flux = _utils.polar_azith_to_vector(
    polar_ang=physical["applied_polar"],
    azith_ang=physical["applied_azith"],
    vec_mag=physical["applied_strength"],
  )
  # Smooth over depth so the field varies continuously with insertion.
  applied_mag_flux = field_filter @ raw_applied_mag_flux
  return remnant_mag_flux, applied_mag_flux

### Init design

In [ ]:
# Axial half-strength remnant, weak field normal to t0 at azimuth -90 deg.
z_axis = np.array([0.0, 0.0, 1.0])
normal_seed = np.cross(entry_tangent, z_axis)
field_normal = normal_seed / np.linalg.norm(normal_seed)
remnant_start = jnp.broadcast_to(0.5 * max_remnant_flux * entry_tangent, (num_elems, 3))
applied_start = jnp.broadcast_to(
  0.05 * max_applied_flux * field_normal, (num_load_steps, 3)
)

remnant_polar, remnant_azith, remnant_strength = _utils.vector_to_polar_azith(
  remnant_start
)
applied_polar, applied_azith, applied_strength = _utils.vector_to_polar_azith(
  applied_start
)
start_physical = {
  "remnant_polar": remnant_polar,
  "remnant_azith": remnant_azith,
  "remnant_strength": remnant_strength,
  "applied_polar": applied_polar,
  "applied_azith": applied_azith,
  "applied_strength": applied_strength,
}
initial_design = layout.to_flat(start_physical)

### Optimization objective

In [ ]:
def compute_net_wall_force(forces: jnp.ndarray) -> jnp.ndarray:
  """Sums the wall force magnitude over all nodes and load steps."""
  squared = jnp.einsum("knd,knd->kn", forces, forces)
  return jnp.sum(_utils.safe_sqrt(x=squared))


def compute_objective(
  design_vars: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
  """Returns the net wall force (1,) [N], its gradient (n, 1), and disp (K, dofs)."""

  def objective_fn(design_vars: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    remnant_mag_flux, applied_mag_flux = design_to_fluxes(design_vars=design_vars)
    _, displacement_history = problem.solve(
      u0=u0,
      shear_moduli=shear_moduli,
      remnant_mag_flux=remnant_mag_flux,
      applied_mag_flux=applied_mag_flux,
    )
    forces, _ = _contact.compute_wall_forces_and_moments(
      rod=rod,
      obstacle=obstacle,
      displacements=displacement_history,
    )
    return compute_net_wall_force(forces=forces), displacement_history

  (objective, displacement_history), grad_objective = jax.value_and_grad(
    objective_fn, has_aux=True
  )(design_vars)
  return objective.reshape(1), grad_objective.reshape(-1, 1), displacement_history


objective_and_grad = jax.jit(compute_objective)

### MMA settings

In [ ]:
max_epochs = 150
plot_interval = 3
normalize_epochs = (0, 10)
depth_mm = 1e3 * insertion_depths
checkpoint_file = pathlib.Path(f"crops/data/results/{case_name}_mma_ckpt.npz")
checkpoint_file.parent.mkdir(parents=True, exist_ok=True)
temp_file = checkpoint_file.with_name(f"{checkpoint_file.stem}_tmp.npz")
debug_file = checkpoint_file.with_name(f"{case_name}_debug.npz")

num_design_var = layout.num_design_var
lower_bound = np.zeros((num_design_var, 1))
upper_bound = np.ones((num_design_var, 1))
mma_params = _mma.MMAParams(
  max_iter=max_epochs,
  kkt_tol=1e-8,
  step_tol=1e-8,
  move_limit=1e-2,
  num_design_var=num_design_var,
  num_cons=1,
  lower_bound=lower_bound,
  upper_bound=upper_bound,
)
init_design_var = np.asarray(initial_design).reshape(-1, 1)
mma_state = _mma.init_mma(
  init_design_var=init_design_var,
  mma_params=mma_params,
)
# No constraints, so MMA gets one inactive dummy.
dummy_cons = np.full((1, 1), -1.0)
dummy_grad_cons = np.zeros((1, num_design_var))

net_wall_force_history = []
design_history = []

In [ ]:
objective_and_grad = jax.jit(compute_objective)  # re-jit, the old trace froze γ
resume = False
if resume:
  checkpoint = np.load(checkpoint_file)
  mma_state = _mma.MMAState.from_array(
    state_array=checkpoint["mma_state"],
    num_design_var=num_design_var,
  )
  objective_scale = float(checkpoint["objective_scale"])
  net_wall_force_history = list(checkpoint["net_wall_force_history"])
  design_history = list(checkpoint["design_history"])
  start_force_over_length = checkpoint["start_force_over_length"]

In [ ]:
while mma_state.epoch < max_epochs and not mma_state.is_converged:
  epoch = mma_state.epoch
  design_vars = jnp.asarray(mma_state.x).reshape(-1)

  epoch_start = time.perf_counter()
  objective, grad_objective, displacement_history = objective_and_grad(
    design_vars=design_vars,
  )

  # Stop before a non-finite objective or gradient reaches MMA.
  grad_blocks = layout.split(flat=grad_objective[:, 0])
  non_finite_grad = {
    name: int(jnp.sum(~jnp.isfinite(block))) for name, block in grad_blocks.items()
  }
  if not np.isfinite(objective).all() or any(non_finite_grad.values()):
    print(f"epoch {epoch}: objective {float(objective[0])}")
    print(f"non-finite gradient entries per block {non_finite_grad}")
    np.savez(
      debug_file,
      design_vars=np.asarray(design_vars),
      grad_objective=np.asarray(grad_objective),
      displacement_history=np.asarray(displacement_history),
    )
    break

  # Normalize by the objective at these epochs, held as a constant.
  if epoch in normalize_epochs:
    objective_scale = float(objective[0])
  normalized_objective = np.asarray(objective, dtype=np.float64) / objective_scale
  normalized_grad = np.asarray(grad_objective, dtype=np.float64) / objective_scale
  mma_state = _mma.update_mma(
    mma_state=mma_state,
    mma_params=mma_params,
    obj=normalized_objective,
    grad_obj=normalized_grad,
    cons=dummy_cons,
    grad_cons=dummy_grad_cons,
  )
  time_epoch = time.perf_counter() - epoch_start

  if not np.isfinite(mma_state.x).all():
    print(f"epoch {epoch}: MMA returned a non-finite design")
    break

  net_wall_force_history.append(float(objective[0]))
  design_history.append(np.asarray(design_vars))
  grad_max = float(np.abs(normalized_grad).max())
  print(
    f"epoch {epoch:3d}  J {float(normalized_objective[0]):.4f}  "
    f"|grad|max {grad_max:.2e}  {time_epoch:.1f} s"
  )

  if epoch % plot_interval == 0:
    forces, _ = _contact.compute_wall_forces_and_moments(
      rod=rod,
      obstacle=obstacle,
      displacements=displacement_history,
    )
    force_over_length = np.linalg.norm(forces, axis=-1).sum(axis=1)
    if epoch == 0:
      start_force_over_length = force_over_length
    relative_history = np.asarray(net_wall_force_history) / net_wall_force_history[0]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(5.0, 3.5))
    ax.plot(relative_history, color="k")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Net wall force / epoch 0")

    fig, ax = plt.subplots(figsize=(5.0, 3.5))
    ax.plot(depth_mm, 1e3 * start_force_over_length, label="epoch 0")
    ax.plot(depth_mm, 1e3 * force_over_length, label=f"epoch {epoch}")
    ax.set_xlabel(r"Insertion depth $d$ [mm]")
    ax.set_ylabel(r"Wall force over length [mN]")
    ax.legend()
    plt.pause(1e-6)

  # The saved MMA state already holds the next design, so a resume continues from it.
  checkpoint = {
    "mma_state": mma_state.to_array(),
    "objective_scale": objective_scale,
    "net_wall_force_history": np.asarray(net_wall_force_history),
    "design_history": np.asarray(design_history),
    "design_vars": np.asarray(design_vars),
    "displacement_history": np.asarray(displacement_history),
    "start_force_over_length": start_force_over_length,
  }
  np.savez(temp_file, **checkpoint)
  temp_file.replace(checkpoint_file)

In [ ]:
import dataclasses

import imageio.v2 as imageio
import polyscope as ps

In [ ]:
checkpoint = np.load(checkpoint_file)
start_design = jnp.asarray(checkpoint["design_history"][0])
final_displacement_history = checkpoint["displacement_history"]  # last evaluated design

start_remnant_flux, start_applied_flux = design_to_fluxes(design_vars=start_design)
_, start_displacement_history = problem.solve(
  u0=u0,
  shear_moduli=shear_moduli,
  remnant_mag_flux=start_remnant_flux,
  applied_mag_flux=start_applied_flux,
)
_, passive_displacement_history = problem.solve(
  u0=u0,
  shear_moduli=shear_moduli,
)

start_forces, _ = _contact.compute_wall_forces_and_moments(
  rod=rod,
  obstacle=obstacle,
  displacements=start_displacement_history,
)
start_peak = float(np.linalg.norm(start_forces, axis=-1).max())
force_range = (0.0, start_peak)  # one color scale for every gif

In [ ]:
frame_interval = 10
contact_radii = np.asarray(rod.contact_radii)
interior_radii = 0.5 * (contact_radii[:-1] + contact_radii[1:])
node_radii = np.concatenate([contact_radii[:1], interior_radii, contact_radii[-1:]])

_viz.init_scene_ps()
static_vessel = dataclasses.replace(obstacle, movement=None)
_viz.plot_obstacle_ps(
  obstacle=static_vessel,
  step_idx=0,
  name="vessel",
)
zero_coords = np.zeros((rod.num_nodes, 3))
zero_field = np.zeros(num_elems)
_viz.plot_rod_centerline_ps(
  node_coords=zero_coords,
  elem_field=zero_field,
  elem_field_val_range=force_range,
  cmap="viridis",
  name="rod",
  node_radii=node_radii,
)

# Look along the centerline's best-fit plane normal.
center = centerline.mean(axis=0)
_, _, principal = np.linalg.svd(centerline - center)
span = float(np.ptp(centerline, axis=0).max())
longest_axis = np.argmax(np.abs(principal[0]))
up_axis = ("x_up", "y_up", "z_up")[longest_axis]
camera_location = tuple(center + 1.6 * span * principal[2])
camera_target = tuple(center)
ps.set_up_dir(up_axis)
ps.look_at(camera_location=camera_location, target=camera_target)

In [ ]:
def render_insertion_gif(
  displacement_history: jnp.ndarray,
  gif_path: pathlib.Path,
) -> None:
  """Renders the insertion in the vessel frame, colored by wall force, as a gif."""
  node_coords, _, _ = _mesher.extract_coords_and_director_from_deformation(
    rod=rod,
    u=displacement_history,
  )
  forces, _ = _contact.compute_wall_forces_and_moments(
    rod=rod,
    obstacle=obstacle,
    displacements=displacement_history,
  )
  force_magnitude = np.linalg.norm(forces, axis=-1)
  elem_force = 0.5 * (force_magnitude[:, :-1] + force_magnitude[:, 1:])

  frame_dir = gif_path.with_suffix("")
  frame_dir.mkdir(parents=True, exist_ok=True)
  last_step = displacement_history.shape[0] - 1
  frame_steps = [*range(0, last_step, frame_interval), last_step]
  frame_files = []
  for step in frame_steps:
    obj_trans, obj_rot_mat = obstacle.pose_at_step(step_idx=step)
    vessel_frame_coords = _utils.affine_transform(
      coords=node_coords[step],
      translation=obj_trans,
      rotation_matrix=obj_rot_mat,
      direction="GLOBAL_TO_LOCAL",
    )
    frame_coords = np.asarray(vessel_frame_coords)
    _viz.update_rod_centerline_ps(
      node_coords=frame_coords,
      elem_field=elem_force[step],
      name="rod",
    )
    frame_file = frame_dir / f"frame_{step:04d}.png"
    frame_name = str(frame_file)
    ps.screenshot(frame_name, transparent_bg=False)
    frame_files.append(frame_file)

  frames = [imageio.imread(frame_file) for frame_file in frame_files]
  imageio.mimsave(gif_path, frames)

In [ ]:
gif_dir = pathlib.Path(f"crops/data/results/{case_name}")
gif_runs = {
  "passive": passive_displacement_history,
  "start_design": start_displacement_history,
  "final_design": final_displacement_history,
}
for run_name, run_history in gif_runs.items():
  gif_path = gif_dir / f"{run_name}_insertion.gif"
  render_insertion_gif(
    displacement_history=jnp.asarray(run_history),
    gif_path=gif_path,
  )